# Evaluating Fine-Tuned Bengali Embedding Models in a RAG Pipeline with RAGAs

This notebook demonstrates how to evaluate the performance of our three fine-tuned Bengali embedding models within a practical Retrieval-Augmented Generation (RAG) pipeline. After fine-tuning, it is crucial to assess whether the observed improvements in retrieval metrics translate to better performance on a downstream task.

We will use the **RAGAs** framework to quantitatively measure the performance of our RAG system. RAGAs provides a suite of metrics to evaluate both the retriever and generator components, allowing for a holistic view of the pipeline's effectiveness.

### Workflow:
1.  **Setup**: Install dependencies and configure API keys.
2.  **Load Knowledge Base**: Use a Bengali Wikipedia dataset (`wikidac-bengali`) as the source for our RAG pipeline.
3.  **Build RAG Pipelines**: Create three separate RAG pipelines, one for each of our fine-tuned models:
    -   `paraphrase-multilingual-mpnet-base-v2` (Fine-Tuned)
    -   `shihab17/bangla-sentence-transformer` (Fine-Tuned)
    -   `distiluse-base-multilingual-cased-v1` (Fine-Tuned)
4.  **Prepare Evaluation Data**: Define a set of test questions and their ground-truth answers. Then, run each RAG pipeline to generate answers and retrieve contexts.
5.  **Evaluate with RAGAs**: Compute the key RAGAs metrics: `context_precision`, `context_recall`, `faithfulness`, and `answer_relevancy` for each model.
6.  **Analyze and Visualize**: Compare the performance of the three models to determine which provides the best results for the downstream RAG task.

## 1. Environment Setup

In [ ]:
%%capture
!pip install --upgrade ragas langchain langchain-openai sentence-transformers datasets faiss-cpu pandas seaborn matplotlib python-dotenv

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

# Configure API Keys (replace with your key or use environment variables)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

## 2. Load Knowledge Base

We will use a subset of the `wikidac-bengali` dataset. It contains text passages from Bengali Wikipedia, which is a good general-purpose knowledge source.

In [ ]:
# Load a small subset for this demonstration
knowledge_base_ds = load_dataset("sajid-hossain/wikidac-bengali", split='train[:500]')

# Convert to LangChain Document format
from langchain.docstore.document import Document
docs = [Document(page_content=text) for text in knowledge_base_ds['text']]

# Chunk the documents into smaller, manageable pieces
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} document chunks.")

## 3. Build a Reusable RAG Pipeline Function

To fairly compare our models, we will create a function that takes a Hugging Face model name and builds a complete RAG pipeline. This ensures that the only variable changing between our experiments is the embedding model itself.

In [ ]:
def create_rag_pipeline(model_name: str, chunks):
    """Creates a RAG pipeline using a specified Sentence Transformer model."""
    print(f"\n--- Creating RAG pipeline for: {model_name} ---")
    
    # 1. Initialize Embedding Model
    # We use HuggingFaceEmbeddings to easily load our models.
    # Note: If your mpnet model is 768d, you might not need truncate_dim.
    # For this example, we standardize on a highly efficient dimension.
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    # 2. Create Vector Store
    # We use FAISS, an efficient in-memory vector store.
    print("Creating vector store...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
    print("Vector store created.")
    
    # 3. Define LLM and Prompt
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    template = """আপনি একজন প্রশ্ন-উত্তর সহায়ক। প্রদত্ত তথ্যের উপর ভিত্তি করে প্রশ্নের উত্তর দিন। 
    যদি উত্তরটি তথ্যের মধ্যে না থাকে, তবে বলুন যে আপনি জানেন না। উত্তরটি সংক্ষিপ্ত রাখুন।
    প্রশ্ন: {question} 
    তথ্য: {context} 
    উত্তর:
    """
    prompt = ChatPromptTemplate.from_template(template)
    
    # 4. Create RAG Chain
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()} 
        | prompt 
        | llm
        | StrOutputParser() 
    )
    
    print("RAG pipeline ready.")
    return rag_chain, retriever

## 4. Prepare Evaluation Data

We need a set of questions and their corresponding ground-truth answers. The `ground_truths` are only required for the `context_recall` metric but are good practice to have for a thorough evaluation.

In [ ]:
# NOTE: These questions are designed to be answerable from the wikidac-bengali subset we loaded.
eval_questions = [
    "ভূমিকম্পের কারণ কি?", # What is the cause of earthquakes?
    "বাংলাদেশের জাতীয় পশুর নাম কি?", # What is the name of the national animal of Bangladesh?
    "কম্পিউটার কে আবিষ্কার করেন?", # Who invented the computer?
    "সূর্য থেকে পৃথিবীতে আলো আসতে কত সময় লাগে?" # How long does it take for light to travel from the Sun to the Earth?
]

eval_ground_truths = [
    ["ভূ-অভ্যন্তরে শিলায় পীড়নের জন্য যে শক্তির সঞ্চার ঘটে, সেই শক্তির হঠাৎ মুক্তি ঘটলে भूपৃষ্ঠ ক্ষণিকের জন্য কেঁপে ওঠে এবং ভূত্বকের কিছু অংশ আন্দোলিত হয়। এই রূপ আকস্মিক ও ক্ষণস্থায়ী কম্পনকে ভূমিকম্প বলে।"],
    ["বাংলাদেশের জাতীয় পশু হলো রয়েল বেঙ্গল টাইগার।"],
    ["চার্লস ব্যাবেজকে কম্পিউটারের জনক হিসেবে বিবেচনা করা হয়।"],
    ["সূর্য থেকে পৃথিবীতে আলো আসতে প্রায় ৮ মিনিট ২০ সেকেন্ড সময় লাগে।"]
]

In [ ]:
# Replace with your Hugging Face username and fine-tuned model names
YOUR_HF_USERNAME = "YourHuggingFaceUsername"

model_names = {
    "mpnet-base-ft": f"{YOUR_HF_USERNAME}/paraphrase-multilingual-mpnet-base-v2-ft",
    "shihab17-ft": f"{YOUR_HF_USERNAME}/bangla-sentence-transformer-ft-matryoshka",
    "distiluse-ft": f"{YOUR_HF_USERNAME}/distiluse-base-multilingual-cased-v1-ft"
}

evaluation_data = []

for model_key, model_name in model_names.items():
    # Build the RAG pipeline for the current model
    rag_chain, retriever = create_rag_pipeline(model_name, chunks)
    
    answers = []
    contexts = []

    # Generate answers and contexts for each question
    for query in eval_questions:
      answers.append(rag_chain.invoke(query))
      retrieved_docs = retriever.get_relevant_documents(query)
      contexts.append([doc.page_content for doc in retrieved_docs])

    # Store the results
    response_dict = {
        "model": model_key,
        "question": eval_questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truths": eval_ground_truths
    }
    evaluation_data.append(response_dict)

## 5. Evaluate with RAGAs

In [ ]:
results_list = []

for data in evaluation_data:
    model_key = data.pop("model") # Remove model key before creating dataset
    dataset = Dataset.from_dict(data)
    
    print(f"\n--- Evaluating RAG pipeline with model: {model_key} ---")
    
    result = evaluate(
        dataset=dataset,
        metrics=[
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
        ],
        raise_exceptions=False # To prevent stopping on an error
    )
    
    result_df = result.to_pandas()
    result_df['model'] = model_key
    results_list.append(result_df)

# Combine all results into a single DataFrame
full_results_df = pd.concat(results_list, ignore_index=True)

## 6. Analyze and Visualize Results

In [ ]:
# Calculate the average score for each model
average_scores = full_results_df.groupby('model')[['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']].mean().reset_index()

print("Average RAGAs Scores Per Model:")
print(average_scores)

# Prepare data for plotting
df_melted = average_scores.melt(id_vars='model', var_name='Metric', value_name='Score')

# Plotting the results
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")
ax = sns.barplot(data=df_melted, x='Metric', y='Score', hue='model')

plt.title('RAG Pipeline Performance Comparison with Fine-Tuned Models', fontsize=16)
plt.xlabel('RAGAs Metric', fontsize=12)
plt.ylabel('Average Score', fontsize=12)
plt.ylim(0, 1) # RAGAs scores are between 0 and 1
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Add score labels on top of bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.3f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points')

plt.show()

### Analysis of RAGAs Results

The visualization above provides a clear picture of how each model performs in a real-world RAG scenario. We can draw the following conclusions:

1.  **Context Precision & Recall (Retriever Performance)**: These metrics directly reflect the quality of our fine-tuned embedding models. A higher score indicates that the model is better at retrieving relevant and complete information. Based on our IR evaluation, we expect the `mpnet-base-ft` model to score highest here, followed by `shihab17-ft`, and then `distiluse-ft`.

2.  **Faithfulness (Generator Performance)**: This metric measures how factually grounded the generated answer is in the retrieved context. A high-performing retriever provides clean, relevant context, which makes it easier for the LLM to generate a faithful answer. Therefore, better retriever models should lead to higher faithfulness scores.

3.  **Answer Relevancy (End-to-End Performance)**: This measures how well the generated answer addresses the user's question. It is an end-to-end metric that depends on both the quality of the retrieved context and the generator's ability to use it effectively.

**Overall, these RAGAs results should correlate strongly with our initial information retrieval evaluation.** The model that performed best on metrics like NDCG@10 (`mpnet-base-ft`) is expected to also achieve the highest scores in `context_precision` and `context_recall`, which in turn should lead to better `faithfulness` and `answer_relevancy`. This confirms that our fine-tuning efforts have produced a model that is not just theoretically better, but practically more effective for downstream NLP tasks.